Важно подавать в функции названии с верными файлами: названия могут отличаться, так как часть разметки проводилась вручную (омонимия + некоторые ошибки)



Сбор исправлений предлодений

In [ ]:
! pip install gigachat -q
! pip install openpyxl -q
! pip install openai -q

! pip install Levenshtein -q
! pip install natasha -q
! pip install pymorphy3 -q

from gigachat import GigaChat
from openai import OpenAI
import pandas as pd
import time

from annotator import Annotator

import matplotlib.pyplot as plt
import numpy as np

from sklearn.cluster import KMeans

from collections import Counter

In [ ]:
# запрос на исправление предложения
def correct_sentence(sentence: str) -> str:

    # промпты
    task_1 = "Здравствуйте! Вам нужно будет исправить ошибки в текстах людей, изучающих русский язык. " \
        "При исправлении текста важно не менять смысл написанного. Следует вносить только необходимые исправления, " \
        "даже если полученное в результате предложение звучит не совсем литературно. Во время выполнения, пожалуйста, " \
        "убедитесь, что после исправленного текста нет лишних символов, например, пробелов." \
        "Нужно предоставить только один вариант исправления без каких-либо комментариев или пояснений." \
        f"Исправьте, пожалуйста, предложение:{sentence}"
    
    task_2 = "Здравствуйте! Вам нужно будет исправить ошибки в текстах людей, изучающих русский язык. " \
            "Пожалуйста, перед исправлением прочитайте предложение до конца." \
            "При исправлении текста важно не менять смысл написанного и рассматривать предложение целиком. " \
            "Следует вносить только необходимые исправления, сохраняя исходную структуру предложения, насколько это возможно," \
            "даже если полученное в результате предложение звучит не совсем литературно. Во время выполнения, пожалуйста, " \
            "убедитесь, что после исправленного текста нет лишних символов, например, пробелов. " \
            "Нужно предоставить только один вариант исправления без каких-либо комментариев или пояснений. " \
            f"Исправьте, пожалуйста, предложение: {sentence}"
    
    # запрос 

    with GigaChat(credentials="", # указываем API ключ
                verify_ssl_certs=False) as giga:
        task = task_2 # указываем нужный пропмпт
        response = giga.chat(task)
        corr_sent = response.choices[0].message.content
    return corr_sent

In [ ]:
df = pd.DataFrame(pd.read_excel('wrong_sent.xlsx')) # надо подать файл с ошибочными предложениями
df_g = df.copy()

for i in range(1, 31):
    df_g[f'var _{i}'] = df_g['wrong sentences'].apply(correct_sentence)

df_g.to_excel('correction vars GigaChat (long prompt).xlsx')

In [ ]:
def correct_sentence(sentence: str) -> str:

    # промпты
    task_1 = "Здравствуйте! Вам нужно будет исправить ошибки в текстах людей, изучающих русский язык. " \
        "При исправлении текста важно не менять смысл написанного. Следует вносить только необходимые исправления, " \
        "даже если полученное в результате предложение звучит не совсем литературно. Во время выполнения, пожалуйста, " \
        "убедитесь, что после исправленного текста нет лишних символов, например, пробелов." \
        "Нужно предоставить только один вариант исправления без каких-либо комментариев или пояснений." \
        f"Исправьте, пожалуйста, предложение:{sentence}"
    
    task_2 = "Здравствуйте! Вам нужно будет исправить ошибки в текстах людей, изучающих русский язык. " \
            "Пожалуйста, перед исправлением прочитайте предложение до конца." \
            "При исправлении текста важно не менять смысл написанного и рассматривать предложение целиком. " \
            "Следует вносить только необходимые исправления, сохраняя исходную структуру предложения, насколько это возможно," \
            "даже если полученное в результате предложение звучит не совсем литературно. Во время выполнения, пожалуйста, " \
            "убедитесь, что после исправленного текста нет лишних символов, например, пробелов. " \
            "Нужно предоставить только один вариант исправления без каких-либо комментариев или пояснений. " \
            f"Исправьте, пожалуйста, предложение: {sentence}"
    
    # запрос 

    time.sleep(5)

    client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="", # указывает ваш ключ
    )

    completion = client.chat.completions.create(
    #deepseek/deepseek-r1-0528:free
    #meta-llama/llama-3.3-70b-instruct:free
    model="meta-llama/llama-3.3-70b-instruct", # передаем название модели
    messages=[
        {
        "role": "user",
        "content": task_2
        }
    ]
    )
    return completion.choices[0].message.content

In [ ]:
df_d = df.copy()
df_l = df.copy()

for i in range(1, 31):
    df_d[f'var _{i}'] = df_d['wrong sentences'].apply(correct_sentence)

df_d.to_excel('correction vars Deepseek (long prompt).xlsx')

for i in range(1, 31):
    df_l[f'var _{i}'] = df_l['wrong sentences'].apply(correct_sentence)

df_l.to_excel('correction vars Llama (long prompt).xlsx')

Рассчет метрик

In [ ]:
# исходные датафреймы
# тут собраны варианты исправлений
g_short = pd.read_excel("3correction vars Gigachat (short prompt).xlsx", index_col=0)
g_long = pd.read_excel("correction vars Gigachat (long prompt).xlsx", index_col=0)

d_short = pd.read_excel("correction vars Deepseek (short prompt).xlsx", index_col=0)
d_long = pd.read_excel("correction vars Deepseek (long prompt).xlsx", index_col=0)

l_short = pd.read_excel("correction vars Llama (short prompt).xlsx", index_col=0)
l_long = pd.read_excel("correction vars Llama (long prompt).xlsx", index_col=0)

In [ ]:
# список с id по порядку строк в исходных df
ordered_ids = [28253, 28787, 25289, 11848, 19703, 6744, 14217, 14939, 25820, 21786, 13405, 25414, 
               23043, 4505, 1113, 12736, 32869, 27883, 14686, 21086, 27027, 5652, 12239, 9071,
               6038, 13574, 5141, 9923, 25924, 11803, 5463, 24610, 31297]
len(set(ordered_ids))

g_short['sent id'] = ordered_ids
g_long['sent id'] = ordered_ids

d_short['sent id'] = ordered_ids
d_long['sent id'] = ordered_ids

l_short['sent id'] = ordered_ids
l_long['sent id'] = ordered_ids

In [ ]:
# добавим золотые исправления
# gold correcftions.xlsx -- файл с золотыми исправлениями

gold = pd.read_excel("gold correcftions.xlsx", header=None)[0].tolist()

g_short['gold corr'] = gold
g_long['gold corr'] = gold

d_short['gold corr'] = gold
d_long['gold corr'] = gold

l_short['gold corr'] = gold
l_long['gold corr'] = gold

In [ ]:
# для единобразияя
names = []
for i in range(1, 31):
    names.append(f'var {i}')

g_short = g_short.rename(columns=dict(zip(g_short.columns[1:-2], names)))
g_long = g_long.rename(columns=dict(zip(g_long.columns[1:-2], names)))

d_short = d_short.rename(columns=dict(zip(d_short.columns[1:-2], names)))
d_long = d_long.rename(columns=dict(zip(d_long.columns[1:-2], names)))

l_short = l_short.rename(columns=dict(zip(l_short.columns[1:-2], names)))
l_long = l_long.rename(columns=dict(zip(l_long.columns[1:-2], names)))

In [ ]:
# функция для преобразования исходных df в списки вариантов для каждого предложения
# везде игнорируем пунктуацию

total_df = pd.DataFrame(columns=['model', 'prompt type', 'sent id', 'initial', 'var', 'var count', 'gold corr'])

import re
def check_latin_letters(sentence: str) -> bool:
    return bool(re.search(r'[a-zA-Z]', sentence))

def count_df(df, model_name, prompt_type, total_df):
    for index, row in df.iterrows():
        punctuation = [',', '.', '!', '?', ':', ';', "\'", '\"', '/', '\t']
        vars = dict()
        var_columns = [f'var {i}' for i in range(1, 31)]
        vars_temp = row[var_columns].tolist()
        # убираем всю пунктуацию
        for v in vars_temp:
            v = str(v).strip()

            for sign in punctuation:
                if sign in v:
                    v = v.replace(sign, '')

            # исправляем опечатки, когда латинские буквы в русских словах
            if check_latin_letters(v): 
                v = v.replace('a', 'а')
                v = v.replace('c', 'с')
                v = v.replace('e', 'е')
                v = v.replace('a', 'а')
                v = v.replace('o', 'о')
                v = v.replace('x','х')
                v = v.replace('E','Е')
                v = v.replace('O','О')
                v = v.replace('H','Н')
                v = v.replace('P','Р')
                v = v.replace('p','р')
                v = v.replace('C','С')
            v = v.replace('  ', ' ')
            v = v.replace('   ', ' ')
            v = v.replace(' - ', ' — ')
            v = v.replace('ё', 'е')
            v = v.replace('Ё', 'Е')

            vars.setdefault(v, 0)
            vars[v] += 1

        counts = [int(x) for x in list(vars.values())]
        vars = list(vars.keys())

        gold = row['gold corr']
        for sign in punctuation:
            if sign in gold:
                gold = gold.replace(sign, '')
        gold = gold.replace('  ', ' ')
        gold = gold.replace('   ', ' ')

        wrong = row['wrong sentences']
        for sign in punctuation:
            if sign in wrong:
                wrong = wrong.replace(sign, '')
        wrong = wrong.replace('  ', ' ')
        wrong = wrong.replace('   ', ' ')

        sent_vars = pd.DataFrame({'model': [model_name] * len(vars), 'prompt type': [prompt_type] * len(vars), 'sent id': [row['sent id']] * len(vars),
                                    'initial': [wrong] * len(vars), 'var': vars, 'var count': counts, 'gold corr': [gold] * len(vars)})
        total_df = pd.concat([total_df, sent_vars], ignore_index=True)
    return total_df

In [ ]:
total_df = count_df(g_short, 'Gigachat-2-Lite', 'short', total_df)
total_df = count_df(g_long, 'Gigachat-2-Lite', 'long', total_df)

total_df = count_df(d_short, 'DeepSeek: R1 0528', 'short', total_df)
total_df = count_df(d_long, 'DeepSeek: R1 0528', 'long', total_df)

total_df = count_df(l_short, 'Llama 3.3 70B Instruct', 'short', total_df)
total_df = count_df(l_long, 'Llama 3.3 70B Instruct', 'long', total_df)

total_df.to_excel("total_df (models).xlsx")

# далее омомнимия была размечена вручную

Аннотация

In [ ]:
a = Annotator()

def annotate(orig_sent: str, corr_var: str) -> list: # выдает список словарей с исправлениями
      edits = a.annotate(orig_sent, corr_var)
      ans = []
      for edit in edits:
            edit = getattr(edit, '__str__')()
            edit_with_quotes = edit.replace("Orig:", "'Orig':").replace("Cor:", "'Cor':").replace("Type:", "'Type':")
            edit_dict = eval('{' + edit_with_quotes + '}')
            ans.append(edit_dict)
      return ans

In [ ]:
total_df['annotations'] = total_df.apply(lambda row: annotate(str(row['initial']), str(row['var'])), axis=1)
total_df['gold annotations'] = total_df.apply(lambda row: annotate(str(row['initial']), str(row['gold corr'])), axis=1)

In [ ]:
# сводный список вида [{правильный тип ошмбки, индекс предложения, есть исправление: 1/0, 
# совпало ли с эталонным: 1/0, вариант, аннотация варианта} - для каждого варинта]"

def mark(sent_ind, var: str, count: int, mist: list, gold_mist: list, mistakes: list, model_name, prompt_type, initial, gold_corr):

    # mist -- аннотация варианта
    # gold_mist -- аннотация золотого исправления
    # в mistakes подаем пустой словарь
    # count -- сколько раз был преложен вариант

    for gold_m in gold_mist: # идем по золотой аннотации и проверяем, что в предложенной
        # проблема: может быть исправлено больше ошибок, чем в золотых исправления или, наоборот, меньше
        # ищем  конкретное исправление в золотых исправлениях по orig: start, end
        # ? достаточно пересечения диапазона
        gold_tag = gold_m['Type']
        gold_corr = gold_m['Cor'][-1]
        gold_orig_start = gold_m['Orig'][0]
        gold_orig_end = gold_m['Orig'][1]

        # дефолтный detected_tag -- на случай, если ошибка не исправлена никак
        detected_tag = ''
        var_annotation = ''
        corrected = 0
        right_corr = 0

        # в предложенном варианте испраления могут по порядку не совпадать с золотыми, пробуем найти исправление в предложенных
        for m in mist:
            orig_start = m['Orig'][0]
            orig_end = m['Orig'][1]

            if (orig_start, orig_end) == (gold_orig_start, gold_orig_end) or \
                (gold_orig_start <= orig_start < gold_orig_end) or \
                (orig_start <= gold_orig_start < orig_end): # это и есть проверка visibility
                corrected = 1 # флаг, что исправление присутствует

                var_annotation = m
                detected_tag = m['Type']
                detected_corr = m['Cor'][-1]
                if detected_corr == gold_corr:
                    right_corr = 1 # флаг, что исправлено
        
        # записываем в словарь
        mist_dict = {'model': model_name,
                    'prompt type': prompt_type,
                    'initial': initial,
                    'gold corr': gold_corr,
                    'gold tag': gold_tag,
                    'sent id': sent_ind,
                    'is corrected': corrected * count,
                    'right correction': right_corr * count,
                    'detected tag': detected_tag,
                    'var annotation': var_annotation,
                    'gold annotation': gold_mist,
                    'var': var}

        mistakes.append(mist_dict)

In [ ]:
mistakes = []
for index, row in total_df.iterrows():
    mark(model_name = row['model'],
         prompt_type = row['prompt type'],
        sent_ind=row['sent id'],
        var=row['var'],
        count=row['var count'],
        mist=row['annotations'],
        gold_mist=row['gold annotations'],
        mistakes=mistakes,
        initial = row['initial'],
        gold_corr = row['gold corr']
    )
mistakes_df = pd.DataFrame(mistakes)

In [ ]:
# временный id_tag --> заменим на id_manual_tag (manual tag проставляем вручную)
mistakes_df['id tag'] = mistakes_df.apply(lambda row: f"{row['sent id']}_{row['manual tag']}", axis=1)

In [ ]:
mistakes_df.to_excel("total_mistakes_df(models).xlsx")

Кластеризация

In [ ]:
def clasterize(my_df, k):
    features = my_df[['visibility', 'precision']]

    '''inertias = []
    K_range = range(1, 11)
    for k in K_range:
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans_temp.fit(features)
        inertias.append(kmeans_temp.inertia_)
    
    plt.figure(figsize=(10, 5))
    plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Количество кластеров (k)')
    plt.ylabel('Инерция')
    plt.title('Метод локтя (elbow method)')
    plt.grid(True, alpha=0.3)
    plt.xticks(K_range)
    plt.show()
    
    improvements = []
    for i in range(len(inertias)-1):
        improvement = (inertias[i] - inertias[i+1]) / inertias[i] * 100
        improvements.append(improvement)

    optimal_k = 2
    for i, imp in enumerate(improvements):
        if imp < 20:
            optimal_k = i + 2
            break'''
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(features)
    my_df['cluster num'] = clusters
    
    return my_df

In [ ]:
df = pd.read_excel("total_mistakes_df(models, no omon).xlsx", index_col=0)

df_aggregated = df[df['prompt type'] == 'long'].groupby(['model', 'id tag']).agg(
    sum_is_corrected = ('is corrected', 'sum'),
    sum_right_correction  = ('right correction', 'sum')
)

df_aggregated = df_aggregated[~df_aggregated.index.get_level_values('id tag').str.contains('Extra')]

df_aggregated.to_excel("all_models_mitrics_25.05.xlsx")

In [ ]:
df_aggregated['visibility'] = df_aggregated['sum_is_corrected'] / 30
df_aggregated['precision'] = df_aggregated['sum_right_correction'] / df_aggregated['sum_is_corrected']
df_aggregated['precision'] = df_aggregated['precision'].fillna(0)

df_aggregated = df_aggregated[df_aggregated['sum_is_corrected'] <= 30]

df_aggregated.describe()

In [ ]:
df_g_long = df_aggregated[df_aggregated.index.get_level_values(0) == 'Gigachat-2-Lite']
df_d_long = df_aggregated[df_aggregated.index.get_level_values(0) == 'DeepSeek: R1 0528']
df_l_long = df_aggregated[df_aggregated.index.get_level_values(0) == 'Llama 3.3 70B Instruct']

In [ ]:
df_g_long_clust = df_g_long[df_g_long['visibility'] > 0.3]
df_d_long_clust = df_d_long[df_d_long['visibility'] > 0.3]
df_l_long_clust = df_l_long[df_l_long['visibility'] > 0.3]

df_g_long_bad = df_g_long[df_g_long['visibility'] <= 0.3]
df_d_long_bad = df_d_long[df_d_long['visibility'] <= 0.3]
df_l_long_bad = df_l_long[df_l_long['visibility'] <= 0.3]

In [ ]:
df_g_long_bad['cluster num'] = 3
df_d_long_bad['cluster num'] = 4
df_l_long_bad['cluster num'] = 4

df_g_long_bad.head(2)

In [ ]:
#g_short_metrics = clasterize_3features(g_short_metrics)
g_long_metrics = clasterize(df_g_long_clust, 3)

#d_short_metrics = clasterize_3features(d_short_metrics)
d_long_metrics = clasterize(df_d_long_clust, 4)

#l_short_metrics = clasterize_3features(l_short_metrics)
l_long_metrics = clasterize(df_l_long_clust, 4)

In [ ]:
d_long_metrics = pd.concat([d_long_metrics, df_d_long_bad])
l_long_metrics = pd.concat([l_long_metrics, df_l_long_bad])
g_long_metrics = pd.concat([g_long_metrics, df_g_long_bad])
g_long_metrics

In [ ]:
#g_short_metric.to_excel("g_short_metrics(no omon).xlsx")
g_long_metrics.to_excel("25_g_long_metrics(no omon).xlsx")

#d_short_metrics.to_excel("d_short_metrics(no omon).xlsx")
d_long_metrics.to_excel("25_d_long_metrics(no omon).xlsx")

#l_short_metrics.to_excel("l_short_metrics(no omon).xlsx")
l_long_metrics.to_excel("25_l_long_metrics(no omon).xlsx")

Графики

In [ ]:
df = pd.read_excel('25_g_long_metrics(no omon).xlsx')

plt.figure(figsize=(10, 8))

# Дискретные цвета для кластеров
unique_clusters = sorted(df['cluster num'].unique())
colors = ['#FF6B6B', '#4ECDC4', "#C846D6", '#FFEAA7']

# Считаем наложения
points = list(zip(df['visibility'], df['precision']))
point_counts = Counter(points)

# Рисуем по кластерам
for cluster in unique_clusters:
    cluster_data = df[df['cluster num'] == cluster]
    x_coords = []
    y_coords = []
    sizes = []
    
    # Группируем точки по уникальным координатам для этого кластера
    grouped = cluster_data.groupby(['visibility', 'precision']).size()
    
    for (vis, prec), count_in_cluster in grouped.items():
        total_count = point_counts[(vis, prec)]  # общее количество (включая другие кластеры)
        
        x_coords.append(vis)
        y_coords.append(prec)
        sizes.append(100 + (total_count - 1) * 80)
    
    plt.scatter(x_coords, y_coords, 
               c=[colors[unique_clusters.index(cluster)]], 
               s=sizes, alpha=0.7, 
               edgecolors='black', linewidth=0.5,
               label=f'Cluster {cluster}')

# Добавляем подписи с количеством наложений (просто черные цифры)
for (vis, prec), count in point_counts.items():
    if count > 1:
        plt.annotate(f'{count}', 
                    (vis, prec), 
                    fontsize=11, 
                    ha='center', 
                    va='center',
                    fontweight='bold',
                    color='black')

plt.xlabel('Visibility', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Gigachat-2-Lite', fontsize=14)

# Легенда только с цветами кластеров
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                              markerfacecolor=colors[i], 
                              markersize=10, 
                              label=f'Cluster {cluster}')
                   for i, cluster in enumerate(unique_clusters)]
plt.legend(handles=legend_elements, title='Clusters', loc='best')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_excel('25_d_long_metrics(no omon).xlsx')

plt.figure(figsize=(10, 8))

# Дискретные цвета для кластеров
unique_clusters = sorted(df['cluster num'].unique())
colors = ['#FF6B6B', '#4ECDC4', "#C846D6", '#96CEB4', '#FFEAA7']

# Считаем наложения
points = list(zip(df['visibility'], df['precision']))
point_counts = Counter(points)

# Рисуем по кластерам
for cluster in unique_clusters:
    cluster_data = df[df['cluster num'] == cluster]
    x_coords = []
    y_coords = []
    sizes = []
    
    # Группируем точки по уникальным координатам для этого кластера
    grouped = cluster_data.groupby(['visibility', 'precision']).size()
    
    for (vis, prec), count_in_cluster in grouped.items():
        total_count = point_counts[(vis, prec)]  # общее количество (включая другие кластеры)
        
        x_coords.append(vis)
        y_coords.append(prec)
        sizes.append(100 + (total_count - 1) * 80)
    
    plt.scatter(x_coords, y_coords, 
               c=[colors[unique_clusters.index(cluster)]], 
               s=sizes, alpha=0.7, 
               edgecolors='black', linewidth=0.5,
               label=f'Cluster {cluster}')

# Добавляем подписи с количеством наложений (просто черные цифры)
for (vis, prec), count in point_counts.items():
    if count > 1:
        plt.annotate(f'{count}', 
                    (vis, prec), 
                    fontsize=11, 
                    ha='center', 
                    va='center',
                    fontweight='bold',
                    color='black')

plt.xlabel('Visibility', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('DeepSeek: R1 0528', fontsize=14)

# Легенда только с цветами кластеров
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                              markerfacecolor=colors[i], 
                              markersize=10, 
                              label=f'Cluster {cluster}')
                   for i, cluster in enumerate(unique_clusters)]
plt.legend(handles=legend_elements, title='Clusters', loc='best')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_excel('25_l_long_metrics(no omon).xlsx')

plt.figure(figsize=(10, 8))

# Дискретные цвета для кластеров
unique_clusters = sorted(df['cluster num'].unique())
colors = ['#FF6B6B', '#4ECDC4', "#C846D6", '#96CEB4', '#FFEAA7']

# Считаем наложения
points = list(zip(df['visibility'], df['precision']))
point_counts = Counter(points)

# Рисуем по кластерам
for cluster in unique_clusters:
    cluster_data = df[df['cluster num'] == cluster]
    x_coords = []
    y_coords = []
    sizes = []
    
    # Группируем точки по уникальным координатам для этого кластера
    grouped = cluster_data.groupby(['visibility', 'precision']).size()
    
    for (vis, prec), count_in_cluster in grouped.items():
        total_count = point_counts[(vis, prec)]  # общее количество (включая другие кластеры)
        
        x_coords.append(vis)
        y_coords.append(prec)
        sizes.append(100 + (total_count - 1) * 80)
    
    plt.scatter(x_coords, y_coords, 
               c=[colors[unique_clusters.index(cluster)]], 
               s=sizes, alpha=0.7, 
               edgecolors='black', linewidth=0.5,
               label=f'Cluster {cluster}')

# Добавляем подписи с количеством наложений (просто черные цифры)
for (vis, prec), count in point_counts.items():
    if count > 1:
        plt.annotate(f'{count}', 
                    (vis, prec), 
                    fontsize=11, 
                    ha='center', 
                    va='center',
                    fontweight='bold',
                    color='black')

plt.xlabel('Visibility', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Llama 3.3 70B Instruct', fontsize=14)

# Легенда только с цветами кластеров
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                              markerfacecolor=colors[i], 
                              markersize=10, 
                              label=f'Cluster {cluster}')
                   for i, cluster in enumerate(unique_clusters)]
plt.legend(handles=legend_elements, title='Clusters', loc='best')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()